# Development notebook


In [ ]:
import numpy as np
import pandas as pd
from matplotlib.axes import Axes
from parameteriser import blast_sequence_against_others, plot_distributions
from parameteriser._plot import y_to_data
from parameteriser.brenda.v0 import Brenda


def select_organism(df: pd.DataFrame, organism: str) -> pd.DataFrame:
    return df[df["organism"] == organism].drop(columns=["organism"])


kms, kcats = Brenda().get_kms_and_kcats(ec="4.1.1.39")
kms.head()

In [ ]:
kms_co2 = kms[kms["substrate"] == "CO2"].drop(columns=["substrate"])
kms_co2.head()

kms_org = select_organism(kms_co2, "Nicotiana tabacum")

In [ ]:
# Select some sequence as reference
query = kms_org.iloc[0]["sequence"]

blast = blast_sequence_against_others(query, kms_co2["sequence"])
blast.head()

In [ ]:
# Weighted mean?

best_result: float = kms_co2.loc[blast.index[0], "value"]

_best_ten = blast.iloc[:10]["pident"]
best_of_ten: float = np.average(
    kms_co2.loc[_best_ten.index, "value"],
    weights=_best_ten.values,
)

In [ ]:
def mark_value(ax: Axes, val: float, annotation: str, ymax: float = 0.925):
    ax.axvline(
        val,
        ymin=ymax - 0.05,
        ymax=ymax,
        color="black",
    )
    ax.annotate(
        annotation,
        xy=(val, y_to_data(ax, ymax + 0.025)),
        ha="center",
    )


fig, ax = plot_distributions(
    kms_co2["value"],
    kms_org["value"],
    ec="4.1.1.39",
    substrate="co2",
    organism_name="Nicotiana tabacum",
)
mark_value(
    ax,
    best_result,
    "blast-best",
    ymax=0.375,
)
mark_value(
    ax,
    best_of_ten,
    "blast-avg",
    ymax=0.725,
)